# Rendering mechanisms & stoichiometric structure

`discopt.mkm` objects render themselves in Jupyter (HTML/LaTeX) and expose the **stoichiometric structure** of a mechanism: independent reactions, Horiuti-Tempkin reaction routes (the overall reaction), and conservation laws (the site balance).

## A model renders as a mechanism table

In [1]:
import discopt.mkm as mk
from discopt.mkm.examples import co_oxidation
m, reactor = co_oxidation(T=500.0)
m   # _repr_html_ -> mechanism table

#,reaction,kinetics,type
1,CO + Pt ⇌ CO∗,"A=10000, Ea=0",reversible
2,O2 + 2 Pt ⇌ 2 O∗,"A=10000, Ea=0",reversible
3,CO∗ + O∗ ⇌ CO2 + 2 Pt,"A=1e+08, Ea=0.7",reversible


## A reaction renders as typeset chemistry; `to_latex()` gives source

In [2]:
r = m.reactions[2]
print(r.to_latex())
print(r.to_html())
r   # _repr_latex_ in a notebook

$CO{}^{\ast} + O{}^{\ast} \rightleftharpoons CO_{2} + 2\,Pt$
CO<sup>&lowast;</sup> + O<sup>&lowast;</sup> &#8652; CO<sub>2</sub> + 2&#8201;Pt


Reaction('CO* + O* <=> CO2 + 2 Pt')

In [3]:
print(m.to_latex())   # an align block for a paper

\begin{align}
  CO + Pt &\rightleftharpoons CO{}^{\ast} \\
  O_{2} + 2\,Pt &\rightleftharpoons 2\,O{}^{\ast} \\
  CO{}^{\ast} + O{}^{\ast} &\rightleftharpoons CO_{2} + 2\,Pt
\end{align}


## Stoichiometric matrix

In [4]:
import numpy as np
from discopt.mkm.analysis import stoichiometry as st
nu, species, reactions = st.stoichiometric_matrix(m)
print('species   :', [s.name for s in species])
for j, rxn in enumerate(reactions):
    print(f'  {rxn.name:18s}', nu[:, j].astype(int))

species   : ['CO', 'O2', 'CO2', 'Pt', 'CO*', 'O*']
  CO adsorption      [-1  0  0 -1  1  0]
  O2 dissociation    [ 0 -1  0 -2  0  2]
  surface reaction   [ 0  0  1  2 -1 -1]


## Reaction routes → the overall reaction

A route is a combination of steps that cancels every surface intermediate; what remains is the overall gas-phase reaction.

In [5]:
for sigma, overall in st.reaction_routes(m):
    terms = '  '.join(f'{v:+g} {s.name}' for s, v in overall.items())
    print('stoichiometric numbers', sigma.astype(int), '->', terms)

stoichiometric numbers [2 1 2] -> -2 CO  -1 O2  +2 CO2


## Independence and conservation laws

In [6]:
print('independent reactions:', st.n_independent_reactions(m), 'of', len(m.reactions))
print('conserved quantities :', st.n_conservation_laws(m))
print('element-balanced     :', st.check_element_balance(m) == [])
print()
# composition is inferred from the formula names; conservation laws then
# separate cleanly into the site balance and per-element balances
for label, law in st.conserved_quantities(m).items():
    print(f'  {label:11s}:', ' + '.join(f'{c:g} {s.name}' for s, c in law.items()), '= const')

independent reactions: 3 of 3
conserved quantities : 3
element-balanced     : True

  site:Pt    : 1 Pt + 1 CO* + 1 O* = const
  element:C  : 1 CO + 1 CO2 + 1 CO* = const
  element:O  : 1 CO + 2 O2 + 2 CO2 + 1 CO* + 1 O* = const


## The solved steady state renders too

In [7]:
sol = mk.solve_steady_state(m, reactor)
sol   # coverages + rates of progress

θ[CO∗],0.5852
θ[O∗],0.2314
θ[Ptfree],0.1835
CO + Pt ⇌ CO∗,1.19
O2 + 2 Pt ⇌ 2 O∗,0.5951
CO∗ + O∗ ⇌ CO2 + 2 Pt,1.19


## Irreversible steps

A step declared `irreversible=True` drops its reverse term (no `K_eq` is used) and renders with a single arrow `\u2192` instead of the equilibrium arrow `\u21cc`. Here a reversible adsorption is followed by an irreversible surface reaction.

In [8]:
mi = mk.Model('irreversible_demo', T=500.0)
si = mi.site('*', density=1.0)
A = mi.gas('A', H=0.0, S=0.002); B = mi.gas('B', H=-1.5, S=0.002)
As = mi.adsorbate('A*', site=si, H=-0.6, S=0.0005)
mi.step(A + si >> As, A=1e6, Ea=0.0, name='adsorption')            # reversible
mi.step(As >> B + si, A=1e9, Ea=0.8, irreversible=True, name='reaction')  # irreversible
mi   # the mechanism table marks the step type

#,reaction,kinetics,type
1,A + ∗ ⇌ A∗,"A=1e+06, Ea=0",reversible
2,A∗ → B + ∗,"A=1e+09, Ea=0.8",irreversible


In [9]:
for r in mi.reactions:
    kind = 'irreversible' if r.irreversible else 'reversible  '
    print(f'{r.name:12s} ({kind}):  {r.to_latex()}')
mi.reactions[1]   # renders with a single arrow

adsorption   (reversible  ):  $A + {}^{\ast} \rightleftharpoons A{}^{\ast}$
reaction     (irreversible):  $A{}^{\ast} \rightarrow B + {}^{\ast}$


Reaction('A* -> B + *')